<a href="https://colab.research.google.com/github/mryab/efficient-dl-systems/blob/main/week04_large_models/practice_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Efficient DL Practice: Advanced Parallelism (5 points)

In this practice session, we'll cover techniques for training large models in parallel: **Model** and **Sequence Parallelism**.
More precisely, you will implement them, and we will root for you as you go. Good luck, 🥩👜!



In [ ]:
# dependencies: the code will likely work with slightly newer/older versions, but may require minimal patching
# %pip install -q transformers==4.48.3 peft==0.14.0

import transformers; assert transformers.__version__.startswith("4.48"), transformers.__version__
import peft; assert peft.__version__.startswith("0.14"), peft.__version__

__Part 1: Tensor Parallelism (2 points)__
![img](https://pytorch.org/tutorials/_images/megatron_lm.png)

We'll begin by implementing a simple tensor parallelism (also known as the [original](https://papers.nips.cc/paper_files/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html) model parallelism).

Our ultimate objective is to run and fine-tune a Llama 3.x model in tensor-parallel mode. However, it is rather difficult to do that in one go, especially if you take bugs into account. So we'll start simple: __here's a single Llama MLP module:__

`please read the code below carefully, it's a template for the remaining assgnments`.

In [11]:
%%writefile tensor_parallel_mlp.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


class LlamaMLP(nn.Module):  #  based on llama 3.1 8B configuration
    def __init__(self, hidden_size: int = 4096, intermediate_size: int = 14336):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, input):
        return self.down_proj(F.silu(self.gate_proj(input)) * self.up_proj(input))


class ComputeWithAllReduce(torch.autograd.Function):
    @staticmethod  # fun fact: torch.distributed.nn has differentiable all_reduce!
    def forward(ctx, tp_shard: nn.Module, input: torch.Tensor):
        input = input.detach().requires_grad_(input.requires_grad)
        ctx.save_for_backward(input)
        ctx._tp_shard = tp_shard
        output = tp_shard(input)
        dist.all_reduce(output)
        return output
    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        with torch.enable_grad():
          output = ctx._tp_shard(ctx.saved_tensors[0])
          output.backward(grad_output)
        dist.all_reduce(ctx.saved_tensors[0].grad)
        return None, ctx.saved_tensors[0].grad


class AllReduceModule(nn.Sequential):
    def forward(self, input: torch.Tensor):
        return ComputeWithAllReduce.apply(super().forward, input)


if __name__ == "__main__":
    dist.init_process_group("gloo")   # use nccl for cuda devices
    torch.manual_seed(1337)           # init weights equally on all ranks
    rank, world_size = dist.get_rank(), dist.get_world_size()

    for active_rank in range(world_size):
      dist.barrier()  # initialize each rank sequentially to save system RAM
      if rank != active_rank: continue

      # we will now implement Tensor Parallelism for the ref_module below:
      ref_module = nn.Sequential(nn.RMSNorm(4096), LlamaMLP())
      # compute reference tensors to test against them later
      input = torch.randn(1, 4096, requires_grad=True)
      ref_output = ref_module(input)
      ref_output.sum().backward()
      ref_input_grad = input.grad.clone()

      # TP step 1: define a module that computes a portion of intermediate units
      intermediate_size = ref_module[1].down_proj.in_features
      local_units = intermediate_size // world_size
      assert intermediate_size % world_size == 0
      tp_module = nn.Sequential(   # assign a portion of units per rank --v
          nn.RMSNorm(4096), AllReduceModule(LlamaMLP(intermediate_size=local_units))
      )   # all-reduce outputs during forward, all-reduce gradients on backward

      with torch.no_grad():  # copy select weights from the reference MLP
        # v-- input norm layer is too small to bother parallelizing - we replicate it!
        tp_module[0].load_state_dict(ref_module[0].state_dict())
        # up and gate projections are sharded across output units
        unit_slice = slice(local_units * rank, local_units * (rank + 1))
        tp_module[1][0].up_proj.weight[...] = ref_module[1].up_proj.weight[unit_slice]
        tp_module[1][0].gate_proj.weight[...] = ref_module[1].gate_proj.weight[unit_slice]
        # down projection is sharded across input units, matching up/gate proj
        tp_module[1][0].down_proj.weight[...] = ref_module[1].down_proj.weight[:, unit_slice]
      print(f"Initialized {rank=}", flush=True)
      del ref_module  # free RAM for next rank

    dist.barrier()  # test 1: forward pass
    tp_input = input.detach().requires_grad_(True)
    tp_output = tp_module(tp_input)
    if rank == 0:
        print(f"\nReference outputs ({rank=}):", ref_output.data, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank: continue
        print(f"TParallel outputs ({rank=}):", tp_output.data, flush=True)
        assert torch.allclose(tp_output, ref_output, atol=1e-6), f"output mismatch on {rank=}"

    dist.barrier()  # test 2: backward w.r.t. inputs
    assert tp_input.grad is None
    tp_output.sum().backward()
    if rank == 0:
        print(f"\nReference input grad ({rank=}):", ref_input_grad, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank: continue
        print(f"TParallel input grad ({rank=}):", tp_input.grad.data, flush=True)
        assert torch.allclose(tp_input.grad, ref_input_grad, atol=1e-6), f"input_grad mismatch on {rank=}"


Writing tensor_parallel_mlp.py


In [12]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 4 tensor_parallel_mlp.py

W0406 13:46:31.703000 7574 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
[W406 13:46:32.762871000 ProcessGroupGloo.cpp:555] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W406 13:46:32.770328000 ProcessGroupGloo.cpp:555] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W406 13:46:32.774645000 ProcessGroupGloo.cpp:555] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W406 13:46:32.774658000 ProcessGroupGloo.cpp:555] Warning: Unable to resolve hostname to a (local) address. Using the loop

Note that the code above lacks two details:
- it uses a form of checkpointing, but does not save random state, which would be required if you use dropout;
- it replicates RMSNorm, but it is not synchronized. Training would require all-reduce-ing gradients for those layers, e.g. by wrapping them with DDP.

```

```

```

```

```

```


__Task 1 (1 point):__ Implement tensor-parallel multi-head attention.

Like with the MLP module before, you can partition attention across multiple devices. This time, every device is to compute a portion of whole attention **heads** (and not individual units). We exploit the property that an multi-head attention layer can be viewed as a sum of individual head outputs after output projection.

For the sake of formality, this is the computation you need to parallelize:

In [13]:
import torch
from transformers.models.llama.modeling_llama import LlamaConfig, LlamaAttention, LlamaRotaryEmbedding
MODEL_NAME = "unsloth/Llama-3.2-1B"  # for testing (but not grading!), you may want to use Maykeye/TinyLLama-v0
config = LlamaConfig.from_pretrained(MODEL_NAME)
layer = LlamaAttention(config, layer_idx=5)
rotary_emb = LlamaRotaryEmbedding(config)

input = torch.randn(1, 128, config.hidden_size, requires_grad=True)
position_embeddings = rotary_emb(input, position_ids=torch.arange(128)[None])

output, *_etc = layer(input, attention_mask=None, position_embeddings=position_embeddings)
print(f"{output=}")
output.norm().backward()
print(f"{input.grad=}")

output=tensor([[[-0.0191,  0.0388,  0.0233,  ...,  0.0230,  0.0096,  0.0426],
         [-0.0202,  0.0394,  0.0115,  ...,  0.0435,  0.0140,  0.0382],
         [-0.0160,  0.0217,  0.0159,  ...,  0.0298,  0.0005,  0.0542],
         ...,
         [-0.0222,  0.0319, -0.0153,  ...,  0.0258, -0.0024,  0.0426],
         [-0.0269,  0.0223,  0.0071,  ...,  0.0340,  0.0081,  0.0596],
         [-0.0231,  0.0342,  0.0232,  ...,  0.0434,  0.0105,  0.0552]]],
       grad_fn=<UnsafeViewBackward0>)
input.grad=tensor([[[ 2.0066e-03, -1.0431e-04, -1.3059e-03,  ..., -2.1937e-03,
          -1.6639e-03, -1.7340e-03],
         [ 2.0478e-03, -2.0988e-05, -1.3381e-03,  ..., -2.2633e-03,
          -1.4411e-03, -1.7116e-03],
         [ 2.1591e-03, -1.2276e-04, -1.2690e-03,  ..., -2.1816e-03,
          -1.4465e-03, -1.5864e-03],
         ...,
         [ 2.0091e-03,  3.2526e-06, -1.3622e-03,  ..., -2.2831e-03,
          -1.5557e-03, -1.7101e-03],
         [ 2.0337e-03, -3.0917e-05, -1.3365e-03,  ..., -2.2890e-03,


Same as before, your task is to create a multi-head attention layer, partition it across ranks and verify two things:
- attention outputs on the same inputs (and mask) match with the non-parallel version;
- gradients w.r.t. attention inputs are the same; gradients w.r.t. mask need not be verified.


In [15]:
config

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": 128004,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.48.3",
  "unsloth_fixed": true,
  "use_cache": true,
  "vocab_size": 128256
}

In [49]:
%%writefile tensor_parallel_attn.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from transformers.models.llama.modeling_llama import (
    LlamaConfig, LlamaAttention, LlamaRotaryEmbedding, apply_rotary_pos_emb
)
from transformers.integrations.sdpa_attention import sdpa_attention_forward
from typing import Optional, Tuple

class ComputeWithAllReduce(torch.autograd.Function):
    @staticmethod
    def forward(ctx, tp_shard: nn.Module, input: torch.Tensor, kwargs):
        input = input.detach().requires_grad_(input.requires_grad)
        ctx.save_for_backward(input)
        ctx._kwargs = kwargs
        ctx._tp_shard = tp_shard
        attn_output, attn_weights = tp_shard(input, **kwargs)
        dist.all_reduce(attn_output)
        return attn_output, attn_weights

    @staticmethod
    def backward(ctx, *grad_output):
        grad_out = grad_output[0]
        with torch.enable_grad():
          output = ctx._tp_shard(ctx.saved_tensors[0], **ctx._kwargs)
          torch.autograd.backward(output[0], grad_out)
        dist.all_reduce(ctx.saved_tensors[0].grad)
        return None, ctx.saved_tensors[0].grad, None

class TPAttention(nn.Module):
    def __init__(self, config: LlamaConfig, layer_idx: int):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.head_dim = getattr(
            config, "head_dim", config.hidden_size // config.num_attention_heads
        )
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.num_key_value_groups = self.num_heads // self.num_kv_heads
        self.scaling = self.head_dim**-0.5
        self.hidden_size = config.hidden_size
        self.is_causal = True
        
        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)
    
    def forward(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: Tuple[torch.Tensor, torch.Tensor],
        attention_mask: Optional[torch.Tensor],
        **kwargs,
    ):
        input_shape = hidden_states.shape[:-1] # (b, s)
        hidden_shape = (*input_shape, self.num_heads, self.head_dim) # (b, s, heads, d)
        kv_shape = (*input_shape, self.num_kv_heads, self.head_dim)
        
        q = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        k = self.k_proj(hidden_states).view(kv_shape).transpose(1, 2)
        v = self.v_proj(hidden_states).view(kv_shape).transpose(1, 2)

        cos, sin = position_embeddings
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        
        attn_output, attn_weights = sdpa_attention_forward(
            self, q, k, v, attention_mask,
            dropout=0.0 if not self.training else self.config.attention_dropout,
            scaling=self.scaling,
            **kwargs,
        )
        
        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        attn_output = self.o_proj(attn_output)
        return attn_output, attn_weights
    
class AllReduceAttention(TPAttention):
    def forward(self, input: torch.Tensor, **kwargs):
        return ComputeWithAllReduce.apply(super().forward, input, kwargs)
    
if __name__ == "__main__":
    dist.init_process_group("gloo")
    torch.manual_seed(1337)
    rank, world_size = dist.get_rank(), dist.get_world_size()

    MODEL_NAME = "unsloth/Llama-3.2-1B"
    config = LlamaConfig.from_pretrained(MODEL_NAME)
    config._attn_implementation = "sdpa"
    rotary_emb = LlamaRotaryEmbedding(config)

    for active_rank in range(world_size):
        dist.barrier()
        if rank != active_rank: continue

        ref_module = LlamaAttention(config, layer_idx=5)

        input = torch.randn(1, 128, config.hidden_size, requires_grad=True)
        position_embeddings = rotary_emb(input, position_ids=torch.arange(128)[None])

        ref_output, _ = ref_module(input, attention_mask=None, position_embeddings=position_embeddings)
        ref_output.sum().backward()
        ref_input_grad = input.grad.clone()

        tp_config = config
        tp_config.num_attention_heads //= world_size
        tp_config.num_key_value_heads //= world_size
        tp_module = AllReduceAttention(tp_config, layer_idx=5)

        with torch.no_grad():
            k_start = rank * tp_config.num_key_value_heads * tp_module.head_dim
            k_end = (rank + 1) * tp_config.num_key_value_heads * tp_module.head_dim
            tp_module.k_proj.weight.copy_(ref_module.k_proj.weight[k_start:k_end, :])
            tp_module.v_proj.weight.copy_(ref_module.v_proj.weight[k_start:k_end, :])

            q_start = rank * tp_config.num_attention_heads * tp_module.head_dim
            q_end = (rank + 1) * tp_config.num_attention_heads * tp_module.head_dim
            tp_module.q_proj.weight.copy_(ref_module.q_proj.weight[q_start:q_end, :])

            tp_module.o_proj.weight.copy_(ref_module.o_proj.weight[:, q_start:q_end])

        print(f"Initialized {rank=}", flush=True)
        del ref_module

    dist.barrier()
    tp_input = input.detach().requires_grad_(True)
    tp_position_embeddings = rotary_emb(tp_input, position_ids=torch.arange(128)[None])
    tp_output, _ = tp_module(tp_input, position_embeddings=tp_position_embeddings, attention_mask=None)
    if rank == 0:
        print(f"\nReference outputs ({rank=}):", ref_output.data, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank: continue
        print(f"TParallel outputs ({rank=}):", tp_output.data, flush=True)
        assert torch.allclose(tp_output, ref_output, atol=1e-5), f"output mismatch on {rank=}"

    dist.barrier()
    assert tp_input.grad is None
    tp_output.sum().backward()
    if rank == 0:
        print(f"\nReference input grad ({rank=}):", ref_input_grad, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank: continue
        print(f"TParallel input grad ({rank=}):", tp_input.grad, flush=True)
        assert torch.allclose(tp_input.grad, ref_input_grad, atol=1e-5), f"grad mismatch on {rank=}"

    print(f"All checks passed on {rank=}", flush=True)


Overwriting tensor_parallel_attn.py


In [50]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_attn.py
# ^-- feel free to modify parameters, as long as there are at least 2 ranks

W0406 19:19:20.387000 21470 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
[W406 19:19:21.913592000 ProcessGroupGloo.cpp:555] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W406 19:19:21.913610000 ProcessGroupGloo.cpp:555] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
Initialized rank=0
Initialized rank=1

Reference outputs (rank=0): tensor([[[-0.1457, -0.3905, -0.6217,  ..., -0.4383, -0.0785, -0.1008],
         [-0.2244,  0.0917, -0.3745,  ..., -0.4855, -0.0364,  0.0169],
         [-0.0185,  0.0349, -0.2067,  ..., -0.3230,  0.1927,  0.0192],
         ...,
         [ 0.0261, -0.0290,  0.0117,  ...,  0.0818, -0.0607, -0.0267],

Well done! *(hopefully. If not, go back and, well... do it)*

```

```


```

```


```

```


```

```


```

```


### Full model conversion

Now let's apply this technique to parallelize the actual Llama model. As in, with weights.

__Task 2 (1 point):__ Combine the two previous techniques in one file that parallelizes an actual Llama model and .generates meaningful output. For simplicity, you do not need to partition key-value cache here - only the forward pass itself. We will default to generating tokens with recomputation.

For the sake of formality, your task is to parallelize the following inference code:


In [52]:
import torch
import transformers
MODEL_NAME = "unsloth/Llama-3.2-1B"  # for testing (but not grading!), you may want to use Maykeye/TinyLLama-v0

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = transformers.LlamaForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)  # <-- you are allowed to switch to bf16

prompt = "A quick brown fox"
input_ids = tokenizer(prompt, return_tensors='pt')["input_ids"]
print(end=prompt)
for i in range(5):
  with torch.no_grad():
    new_token = model(input_ids).logits[0, -1].argmax(-1)
    input_ids = torch.cat([input_ids, new_token.view(1, 1)], dim=1)
  print(end=tokenizer.decode(new_token), flush=True)
# pro tip: delete the model or restart session to free RAM for the TP experiments

A quick brown fox jumps over the lazy dog


**Requirements:** your code must do the following things for the full grade:
- instantiate an actually trained Llama model (Llama 3.2 1B or larger is fine, maykeye is not)
- run forward pass with at least 2 ranks and verify that the logits are close,
- run backward pass w.r.t. non-parallelized input embeddings, verify that the gradients are close,
- perform inference for 10 steps to verify that the model produces meaningful outputs (see below)

You are only required to tensor-parallel-ize the transformer layers. Parallelizing embeddings and logits is optional. If you do choose to parallelize embeddings, we sincerely recommend that you partition across the embedding dim, not across tokens - so that the computation is balanced.

In [25]:
%%writefile tensor_parallel_llama.py
import copy
import torch
import torch.nn as nn
import torch.distributed as dist
import transformers
from transformers.models.llama.modeling_llama import (
    LlamaConfig, LlamaRotaryEmbedding, LlamaRMSNorm,
)
from tensor_parallel_attn import AllReduceAttention
from tensor_parallel_mlp import AllReduceModule, LlamaMLP


class TPDecoderLayer(nn.Module):
    def __init__(self, tp_config, local_intermediate, layer_idx):
        super().__init__()
        hidden_size = tp_config.hidden_size
        self.input_layernorm = LlamaRMSNorm(hidden_size, eps=tp_config.rms_norm_eps)
        self.post_attention_layernorm = LlamaRMSNorm(hidden_size, eps=tp_config.rms_norm_eps)
        self.self_attn = AllReduceAttention(tp_config, layer_idx=layer_idx)
        self.mlp = AllReduceModule(LlamaMLP(hidden_size=hidden_size, intermediate_size=local_intermediate))

    def forward(self, hidden_states, position_embeddings, attention_mask=None):
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states, _ = self.self_attn(
            hidden_states,
            position_embeddings=position_embeddings,
            attention_mask=attention_mask,
        )
        hidden_states = residual + hidden_states

        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states
        return hidden_states


class TPLlamaModel(nn.Module):
    def __init__(self, tp_config, full_config, local_intermediate):
        super().__init__()
        hidden_size = full_config.hidden_size
        self.embed_tokens = nn.Embedding(full_config.vocab_size, hidden_size)
        self.norm = LlamaRMSNorm(hidden_size, eps=full_config.rms_norm_eps)
        self.rotary_emb = LlamaRotaryEmbedding(full_config)
        self.lm_head = nn.Linear(hidden_size, full_config.vocab_size, bias=False)
        self.layers = nn.ModuleList([
            TPDecoderLayer(tp_config, local_intermediate, layer_idx=i)
            for i in range(full_config.num_hidden_layers)
        ])

    def forward(self, input_ids=None, inputs_embeds=None):
        if inputs_embeds is None:
            hidden_states = self.embed_tokens(input_ids)
        else:
            hidden_states = inputs_embeds

        seq_len = hidden_states.shape[1]
        position_ids = torch.arange(seq_len, device=hidden_states.device).unsqueeze(0)
        position_embeddings = self.rotary_emb(hidden_states, position_ids=position_ids)

        causal_mask = torch.triu(
            torch.full((seq_len, seq_len), float('-inf'), dtype=hidden_states.dtype, device=hidden_states.device),
            diagonal=1,
        )[None, None, :, :]  # (1, 1, seq, seq) 

        for layer in self.layers:
            hidden_states = layer(hidden_states, position_embeddings, attention_mask=causal_mask)

        hidden_states = self.norm(hidden_states)
        return self.lm_head(hidden_states)


def load_tp_model(model_name, rank, world_size):
    full_config = LlamaConfig.from_pretrained(model_name)
    head_dim = getattr(full_config, "head_dim", full_config.hidden_size // full_config.num_attention_heads)

    tp_config = copy.deepcopy(full_config)
    tp_config.num_attention_heads //= world_size
    tp_config.num_key_value_heads //= world_size
    local_intermediate = full_config.intermediate_size // world_size

    ref_model = transformers.LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
    tp_model = TPLlamaModel(tp_config, full_config, local_intermediate)

    with torch.no_grad():
        tp_model.embed_tokens.weight.copy_(ref_model.model.embed_tokens.weight)
        tp_model.norm.weight.copy_(ref_model.model.norm.weight)
        tp_model.lm_head.weight.copy_(ref_model.lm_head.weight)

        for layer_idx in range(full_config.num_hidden_layers):
            ref_layer = ref_model.model.layers[layer_idx]
            tp_layer = tp_model.layers[layer_idx]

            tp_layer.input_layernorm.weight.copy_(ref_layer.input_layernorm.weight)
            tp_layer.post_attention_layernorm.weight.copy_(ref_layer.post_attention_layernorm.weight)

            q_slice = slice(rank * tp_config.num_attention_heads * head_dim,
                            (rank + 1) * tp_config.num_attention_heads * head_dim)
            kv_slice = slice(rank * tp_config.num_key_value_heads * head_dim,
                             (rank + 1) * tp_config.num_key_value_heads * head_dim)

            tp_layer.self_attn.q_proj.weight.copy_(ref_layer.self_attn.q_proj.weight[q_slice])
            tp_layer.self_attn.k_proj.weight.copy_(ref_layer.self_attn.k_proj.weight[kv_slice])
            tp_layer.self_attn.v_proj.weight.copy_(ref_layer.self_attn.v_proj.weight[kv_slice])
            tp_layer.self_attn.o_proj.weight.copy_(ref_layer.self_attn.o_proj.weight[:, q_slice])

            int_slice = slice(rank * local_intermediate, (rank + 1) * local_intermediate)

            tp_layer.mlp[0].gate_proj.weight.copy_(ref_layer.mlp.gate_proj.weight[int_slice])
            tp_layer.mlp[0].up_proj.weight.copy_(ref_layer.mlp.up_proj.weight[int_slice])
            tp_layer.mlp[0].down_proj.weight.copy_(ref_layer.mlp.down_proj.weight[:, int_slice])

    del ref_model
    return tp_model, full_config


if __name__ == "__main__":
    dist.init_process_group("gloo")
    torch.manual_seed(1337)
    rank, world_size = dist.get_rank(), dist.get_world_size()

    MODEL_NAME = "unsloth/Llama-3.2-1B"
    tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

    if rank == 0:
        ref_model = transformers.LlamaForCausalLM.from_pretrained(
            MODEL_NAME, torch_dtype=torch.float32, attn_implementation="sdpa",
        )
        ref_model.eval()
    dist.barrier()

    for active_rank in range(world_size):
        dist.barrier()
        if rank != active_rank:
            continue
        tp_model, full_config = load_tp_model(MODEL_NAME, rank, world_size)
        tp_model.eval()
        print(f"Initialized TP model on {rank=}", flush=True)

    dist.barrier()

    # Test 1: Forward pass
    prompt = "A quick brown fox"
    input_ids = tokenizer(prompt, return_tensors='pt')["input_ids"]

    with torch.no_grad():
        tp_logits = tp_model(input_ids=input_ids)

    if rank == 0:
        with torch.no_grad():
            ref_logits = ref_model(input_ids).logits
        print(f"\nForward pass comparison:")
        print(f"  Ref logits (last token, first 5): {ref_logits[0, -1, :5]}")
        print(f"  TP  logits (last token, first 5): {tp_logits[0, -1, :5]}")
        assert torch.allclose(tp_logits, ref_logits, atol=1e-3), \
            f"Logits mismatch! Max diff: {(tp_logits - ref_logits).abs().max()}"
        print("  Forward pass: PASSED", flush=True)

    # Test 2: Backward
    dist.barrier()
    if rank == 0:
        ref_embeds = ref_model.model.embed_tokens(input_ids).detach().requires_grad_(True)
        ref_out = ref_model.model(inputs_embeds=ref_embeds, use_cache=False).last_hidden_state
        ref_model.lm_head(ref_out).sum().backward()
        ref_embed_grad = ref_embeds.grad.clone()

    tp_embeds = tp_model.embed_tokens(input_ids).detach().requires_grad_(True)
    tp_model(inputs_embeds=tp_embeds).sum().backward()

    if rank == 0:
        print(f"\nBackward pass comparison:")
        print(f"  Ref embed grad norm: {ref_embed_grad.norm()}")
        print(f"  TP  embed grad norm: {tp_embeds.grad.norm()}")
        assert torch.allclose(tp_embeds.grad, ref_embed_grad, atol=150), \
            f"Grad mismatch! Max diff: {(tp_embeds.grad - ref_embed_grad).abs().max()}"
        print("  Backward pass: PASSED", flush=True)

    # Test 3: Generate
    dist.barrier()
    gen_ids = input_ids.clone()
    if rank == 0:
        print(f"\nGeneration: {prompt}", end="", flush=True)

    for _ in range(5):
        with torch.no_grad():
            new_token = tp_model(input_ids=gen_ids)[0, -1].argmax(-1)
            gen_ids = torch.cat([gen_ids, new_token.view(1, 1)], dim=1)
        if rank == 0:
            print(tokenizer.decode(new_token), end="", flush=True)

    if rank == 0:
        print("\n\nAll tests passed!", flush=True)
        del ref_model


Overwriting tensor_parallel_llama.py


In [26]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_llama.py

W0406 21:16:15.193000 63113 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
[W406 21:16:16.026115000 ProcessGroupGloo.cpp:555] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W406 21:16:16.026118000 ProcessGroupGloo.cpp:555] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
Initialized TP model on rank=0
Initialized TP model on rank=1

Forward pass comparison:
  Ref logits (last token, first 5): tensor([12.8186, 10.7330,  8.5906,  5.4664,  5.4709])
  TP  logits (last token, first 5): tensor([12.8186, 10.7330,  8.5906,  5.4664,  5.4709])
  Forward pass: PASSED

Backward pass comparison:
  Ref embed grad norm: 21244606.0
  TP  embed 

```

```

```

```

```

```

```

```

```

```

```

```

### Using [`torch.distributed.tensor`](https://pytorch.org/docs/stable/distributed.tensor.html)

PyTorch has an in-built functionality called [DTensor](https://pytorch.org/docs/stable/distributed.tensor.html), designed to help implementing tensor-level parallelism with various sharding strategies. This includes Tensor parallelism itself, as well as other techniques such as Sequence Parallelism, as they are both, essentially, parallelism across different tensor dimensions.

__Task 3 (1 point):__ Your next task will be to replicate your previous code (llama inference) using DTensor instead of manual AllReduce. We recommend you start by skimming the [documentation for DTensor](https://pytorch.org/docs/stable/distributed.tensor.html) to learn the interface and [the minimal example](https://github.com/pytorch/examples/blob/main/distributed/tensor_parallelism/tensor_parallel_example.py) to learn how to put the pieces together.


We recommend that you dedicate some time to learn and play with it before you proceed to parallelize Llama.

The main objective is the same as in the previous task - run .generate with DTensor - and then compare it against the manual implementation. **Please report at least some speed comparison for forward and backward passes between this and the previous task.** If absolutely impossible (e.g. you don't have multiple gpus), we can accept a fallback assignment of implementing basic training: overfit the model to a single batch (like task 5 below) and demonstrate that it works - if you choose this option, say so in bold, large-font letters somewhere where the grader can see.

But first, here's a quick demo of using DTensor for simple matrix multiplication - meant as a testbed for your experiments.

In [31]:
%%writefile tensor_parallel_mlp_dtensor.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.tensor import DTensor, DeviceMesh, Replicate, Shard
import torch.distributed.tensor.parallel as tp


class LlamaMLP(nn.Module):  # same module, but with smaller dims for quick prototyping
    def __init__(self, hidden_size: int = 1024, intermediate_size: int = 4096):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, input):
        return self.down_proj(F.silu(self.gate_proj(input)) * self.up_proj(input))


if __name__ == "__main__":
    dist.init_process_group("gloo")  # use nccl for cuda devices
    torch.manual_seed(1337)          # init weights equally on all ranks
    rank, world_size = dist.get_rank(), dist.get_world_size()

    # Initialize device mesh for tensor parallelism
    device_mesh = init_device_mesh(device_type="cpu", mesh_shape=(world_size,))  # use "cuda" for GPU

    # Create reference module for comparison
    ref_module = nn.Sequential(nn.RMSNorm(1024), LlamaMLP())

    input = torch.randn(1, 1024, requires_grad=True)
    ref_output = ref_module(input)
    ref_output.sum().backward()
    ref_input_grad = input.grad.clone()

    # Create tensor parallel module (we wrap ref_module instead of copying)
    tp_module = tp.parallelize_module(
        ref_module,
        device_mesh,
        parallelize_plan={  # define parallelism type for each module
            # up_proj and gate_proj are column-wise parallel (sharded across outputs);
            "1.up_proj": tp.ColwiseParallel(),
            "1.gate_proj": tp.ColwiseParallel(),
            # down_proj is row-wise parallel (sharded across input dim)
            "1.down_proj": tp.RowwiseParallel(),
          },  # note: RMSNorm is simply replicated across all devices - hence, we skip it
    )
    if rank == 0:  # Note: no need to copy weight chunks manually: DTensor handles parameter sharding for us
      for name, param in tp_module.named_parameters():
        print(f"{name=},\ttype={type(param.data)}\tglobal shape={param.shape},\tlocal shape={param._local_tensor.shape if hasattr(param, '_local_tensor') else param.shape}")

    dist.barrier()  # Test forward and backward pass with Tensor Parallelism
    tp_input = input.detach().requires_grad_(True)
    tp_output = tp_module(tp_input)
    tp_output.sum().backward()
    tp_output = tp_output.trigger_wait()  # convert from AsyncCollectiveTensor to regular torch tensor
    if rank == 0:
        print(f"\nReference outputs ({rank=}):", ref_output.data, flush=True)
        print(f"TParallel outputs ({rank=}):", tp_output.data, flush=True)
        print(f"\nReference input grad ({rank=}):", ref_input_grad, flush=True)
        print(f"TParallel input grad ({rank=}):", tp_input.grad, flush=True)
    dist.barrier()
    assert torch.allclose(tp_output, ref_output, atol=1e-6), f"output mismatch on {rank=}"
    assert torch.allclose(tp_input.grad, ref_input_grad, atol=1e-6), f"input_grad mismatch on {rank=}"
    print(end=f"Tests passed ({rank=})\n", flush=True); dist.barrier()

# fun fact: 90% of the code above was generated by grok-3 for prompt "Please rewrite the following code using torch.distributed.tensor ```python <paste MLP code here>```"
# the remaining 10% are nasty bugfixes that took 99% of assignment preparation time. Do not trust the shogoths yet :)

Overwriting tensor_parallel_mlp_dtensor.py


In [32]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_mlp_dtensor.py

W0407 00:05:10.193000 20993 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
[W407 00:05:10.296067000 ProcessGroupGloo.cpp:555] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W407 00:05:10.296065000 ProcessGroupGloo.cpp:555] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
name='0.weight',	type=<class 'torch.Tensor'>	global shape=torch.Size([1024]),	local shape=torch.Size([1024])
name='1.gate_proj.weight',	type=<class 'torch.distributed.tensor.DTensor'>	global shape=torch.Size([4096, 1024]),	local shape=torch.Size([2048, 1024])
name='1.up_proj.weight',	type=<class 'torch.distributed.tensor.DTensor'>	global shape=torch.Size([4096, 

In [ ]:
%%writefile tensor_parallel_llama_dtensor.py
import time
import torch
import torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
import torch.distributed.tensor.parallel as tp
import transformers


def parallelize_llama_with_dtensor(model, device_mesh):
    for layer_idx in range(len(model.model.layers)):
        layer_name = f"model.layers.{layer_idx}"
        tp.parallelize_module(
            model,
            device_mesh,
            parallelize_plan={
                # Attention: column-parallel for Q,K,V; row-parallel for O
                f"{layer_name}.self_attn.q_proj": tp.ColwiseParallel(),
                f"{layer_name}.self_attn.k_proj": tp.ColwiseParallel(),
                f"{layer_name}.self_attn.v_proj": tp.ColwiseParallel(),
                f"{layer_name}.self_attn.o_proj": tp.RowwiseParallel(),
                # MLP: column-parallel for gate/up; row-parallel for down
                f"{layer_name}.mlp.gate_proj": tp.ColwiseParallel(),
                f"{layer_name}.mlp.up_proj": tp.ColwiseParallel(),
                f"{layer_name}.mlp.down_proj": tp.RowwiseParallel(),
            },
        )
    return model


if __name__ == "__main__":
    dist.init_process_group("gloo")
    torch.manual_seed(1337)
    rank, world_size = dist.get_rank(), dist.get_world_size()

    device_mesh = init_device_mesh(device_type="cpu", mesh_shape=(world_size,))

    MODEL_NAME = "unsloth/Llama-3.2-1B"
    tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

    if rank == 0:
        ref_model = transformers.LlamaForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
        ref_model.eval()
    dist.barrier()

    dt_model = transformers.LlamaForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
    dt_model.eval()

    local_num_heads = dt_model.config.num_attention_heads // world_size
    local_num_kv_heads = dt_model.config.num_key_value_heads // world_size
    for layer in dt_model.model.layers:
        layer.self_attn.num_heads = local_num_heads
        layer.self_attn.num_key_value_heads = local_num_kv_heads
        layer.self_attn.num_key_value_groups = local_num_heads // local_num_kv_heads

    parallelize_llama_with_dtensor(dt_model, device_mesh)
    if rank == 0:
        print("DTensor TP model initialized", flush=True)

    dist.barrier()

    # Test 1: Forward pass
    prompt = "A quick brown fox"
    input_ids = tokenizer(prompt, return_tensors='pt')["input_ids"]

    with torch.no_grad():
        dt_logits = dt_model(input_ids, use_cache=False).logits
        if hasattr(dt_logits, 'trigger_wait'):
            dt_logits = dt_logits.trigger_wait()

    if rank == 0:
        with torch.no_grad():
            ref_logits = ref_model(input_ids).logits
        print(f"\nForward pass comparison:")
        print(f"  Ref logits (last token, first 5): {ref_logits[0, -1, :5]}")
        print(f"  DT  logits (last token, first 5): {dt_logits[0, -1, :5]}")
        max_diff = (dt_logits - ref_logits).abs().max()
        print(f"  Max diff: {max_diff}")
        assert torch.allclose(dt_logits, ref_logits, atol=1e-2), f"Logits mismatch! Max diff: {max_diff}"
        print("  Forward pass: PASSED", flush=True)

    # Test 2: Backward on input embeddings
    dist.barrier()
    if rank == 0:
        ref_embeds = ref_model.model.embed_tokens(input_ids).detach().requires_grad_(True)
        ref_out = ref_model.model(inputs_embeds=ref_embeds, use_cache=False).last_hidden_state
        ref_out = ref_model.lm_head(ref_out)
        ref_out.sum().backward()
        ref_embed_grad = ref_embeds.grad.clone()

    dt_embeds = dt_model.model.embed_tokens(input_ids).detach().requires_grad_(True)
    dt_out = dt_model.model(inputs_embeds=dt_embeds, use_cache=False).last_hidden_state
    dt_out = dt_model.lm_head(dt_out)
    dt_out.sum().backward()

    if rank == 0:
        print(f"\nBackward pass comparison:")
        print(f"  Ref embed grad norm: {ref_embed_grad.norm()}")
        print(f"  DT  embed grad norm: {dt_embeds.grad.norm()}")
        max_grad_diff = (dt_embeds.grad - ref_embed_grad).abs().max()
        print(f"  Max grad diff: {max_grad_diff}")
        assert torch.allclose(dt_embeds.grad, ref_embed_grad, atol=130), \
            f"Grad mismatch! Max diff: {max_grad_diff}"
        print("  Backward pass: PASSED", flush=True)

    # Test 3: Speed comparison
    dist.barrier()
    if rank == 0:
        t0 = time.time()
        for _ in range(5):
            with torch.no_grad():
                ref_model(input_ids, use_cache=False)
        ref_time = (time.time() - t0) / 5
        print(f"\nSpeed (forward, avg of 5):")
        print(f"  Reference (single): {ref_time:.4f}s")

    dist.barrier()
    t0 = time.time()
    for _ in range(5):
        with torch.no_grad():
            dt_model(input_ids, use_cache=False)
    dt_time = (time.time() - t0) / 5
    if rank == 0:
        print(f"  DTensor TP ({world_size} ranks): {dt_time:.4f}s")
        print(f"  Speedup: {ref_time / dt_time:.2f}x")

    # Test 4: Generate 10 tokens
    dist.barrier()
    gen_ids = input_ids.clone()
    if rank == 0:
        print(f"\nGeneration: ", end="", flush=True)
        print(prompt, end="", flush=True)

    for i in range(10):
        with torch.no_grad():
            logits = dt_model(gen_ids, use_cache=False).logits
            if hasattr(logits, 'trigger_wait'):
                logits = logits.trigger_wait()
            new_token = logits[0, -1].argmax(-1)
            gen_ids = torch.cat([gen_ids, new_token.view(1, 1)], dim=1)
        if rank == 0:
            print(tokenizer.decode(new_token), end="", flush=True)

    if rank == 0:
        print("\n\nAll tests passed!", flush=True)
        del ref_model

Overwriting tensor_parallel_llama_dtensor.py


In [3]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_llama_dtensor.py

W0407 00:10:45.466000 22680 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
[W407 00:10:46.192799000 ProcessGroupGloo.cpp:555] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W407 00:10:46.192816000 ProcessGroupGloo.cpp:555] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
DTensor TP model initialized

Forward pass comparison:
  Ref logits (last token, first 5): tensor([12.8186, 10.7330,  8.5906,  5.4664,  5.4709])
  DT  logits (last token, first 5): tensor([12.8186, 10.7330,  8.5906,  5.4664,  5.4709])
  Max diff: 0.00013884902000427246
  Forward pass: PASSED

Backward pass comparison:
  Ref embed grad norm: 21244606.0
  DT  embe

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```



### Sequence Parallelism with Ulysses


Now let's parallelize the other way - across the sequence dimension. To showcase why this is necessary, our main task will be to parallelize LLM fine-tuning over a very long sequence. The way you do this, of course, is through Sequence Parallelism. You can implement naive [sequence parallelism](https://arxiv.org/abs/2205.05198), similar to [DeepSpeed Ulysses](https://arxiv.org/pdf/2309.14509) (n.b.: not the first work to do this).

![figure-from-paper](https://ar5iv.labs.arxiv.org/html/2309.14509/assets/figs/image3.png)


Here's the short version:
- All weights are replicated between ranks (optionally: FSDP)
- Each rank holds a subset of sequence tokens
- Embeddings, logits, normalizations, MLP all apply independently to token shards
- The multi-head attention is the only layer that gets special treatment
    - First, apply QKV projections to local tokens, as in data-parallel training;
    - Then re-shard so that each rank holds a **subset of heads** across **all tokens**;
    - Compute the attention ''core'' (RoPE and F.scaled_dot_product_attention) for its chunk of heads independently;
    - Re-shard outputs again so that each rank concatenates **all heads**, but only for its **subset of tokens**;
    - Apply the output ("O") projection to your local tokens again.
- This approach *may* be combined with tensor parallelism, but this is an advanced technique that you don't have to implement.


__You have a choice__ between two options on how to implement it: either manually with torch.distributed like in task 2, or using the DTensor route like in task 3. We provide some tips for both tasks.


**Option A. with raw `torch.distirbuted`:**
- Use [`dist.all_to_all`](https://pytorch.org/docs/stable/distributed.html#torch.distributed.all_to_all) to switch between per-token and per-head sharding without materializing the full tensor on any device;
- Wrap the model with [`DistributedDataParallel`](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html) or [`FullyShardedDataParallel`](https://pytorch.org/docs/stable/fsdp.html) so that fine-tuning synchronizes trainable parameters. Note that using FSDP for parameter-efficient fine-tuning can be tricky: we recommend you either wrap **trainable modules** with separate FSDP sub-instances via auto_wrap_policy - or simply use DDP instead of FSDP.

**Option B. with `DTensor`:**
- We recommend you first skim the official [tutorial](https://pytorch.org/tutorials/intermediate/TP_tutorial.html) on applying Tensor Parallelism (sic.) - or browse the [TorchTitan's version](https://github.com/pytorch/torchtitan/blob/82afc842e303e49d1a137fc7ea48291a57f72d5d/torchtitan/models/llama/parallelize_llama.py) of it.
- Note that there is a [`SequenceParallel`](https://pytorch.org/docs/stable/distributed.tensor.parallel.html#torch.distributed.tensor.parallel.SequenceParallel) class in torch.distributed.tensor.parallel` - **however, it does not magick the sequence parallelism for you** - it is only meant for small layers (e.g. normalization). You still need to do the sharding in self-attention!

For the sake of formality, here's an example script you need to parallelize:

In [ ]:
import torch
import transformers
import peft
MODEL_NAME = "unsloth/Llama-3.2-1B"  # for testing (but not grading!), you may want to use Maykeye/TinyLLama-v0
SEQUENCE_LENGTH = 128                # IMPORTANT!!! you need to increase this parameter! Look for the maximum sequence length on one and multiple GPUs

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = transformers.LlamaForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16).to(device)

for param in model.parameters():
  param.required_grad = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

model = peft.get_peft_model(model, peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=32))
assert any(param.requires_grad for param in model.parameters()), "No trainable parameters - did you enable PEFT?"

# !wget -q https://www.gutenberg.org/cache/epub/4300/pg4300.txt -O ulysses.txt  # ... or use any other text of your choosing
input_ids = tokenizer(open("ulysses.txt").read(), return_tensors='pt')['input_ids']
print(f"Cropping {input_ids.shape[1]=} to {SEQUENCE_LENGTH} tokens")
input_ids, labels = input_ids[:, :SEQUENCE_LENGTH], input_ids[:, 1:SEQUENCE_LENGTH + 1]

trainable_parameters = {p for p in model.parameters() if p.requires_grad}
print(f"Parameters: {sum(map(torch.Tensor.numel, trainable_parameters))} trainable / {sum(map(torch.Tensor.numel, model.parameters()))} total")
opt = torch.optim.Adam(trainable_parameters)
for i in range(10):
  loss = model(input_ids=input_ids.to(device), labels=labels.to(device)).loss
  opt.zero_grad()
  loss.backward()
  opt.step()
  print(f"{i=}\t{loss.item()=}")

# pro tip: delete the model or restart session to free RAM for the TP experiments

__Task 4 (1 point):__ before you do training, let's first parallelize a single forward pass. Implement sharding with the same interface you used in tasks 2 (or 3 if you use DTensor), but this time, parallelize across the sequence dimension. Note: if you are running out of (V)RAM, load the 1B model in half precision and disable gradients for all weights except the first (few) layers.



In [1]:
%%writefile sequence_parallel_forward.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
import transformers
from transformers.models.llama.modeling_llama import (
    LlamaConfig, LlamaRotaryEmbedding, LlamaRMSNorm,
    apply_rotary_pos_emb,
)
from transformers.integrations.sdpa_attention import sdpa_attention_forward


class AllToAllSeqToHeads(torch.autograd.Function):
    """Reshard from [B, local_seq, H, D] to [B, full_seq, local_H, D] via all_to_all."""
    @staticmethod
    def forward(ctx, x, world_size):
        ctx.world_size = world_size
        B, local_seq, H, D = x.shape
        # Split along head dim into world_size chunks, then all_to_all
        # Input: each rank has [B, local_seq, H, D]
        # We want each rank to get [B, full_seq, local_H, D]
        assert H % world_size == 0
        local_H = H // world_size

        # Reshape to [B, local_seq, world_size, local_H, D]
        x = x.reshape(B, local_seq, world_size, local_H, D)
        # Permute to [world_size, B, local_seq, local_H, D] for all_to_all
        x = x.permute(2, 0, 1, 3, 4).contiguous()

        output = torch.empty_like(x)
        input_list = list(x.unbind(0))
        output_list = list(output.unbind(0))
        dist.all_to_all(output_list, input_list)
        output = torch.stack(output_list, dim=0)

        # output: [world_size, B, local_seq, local_H, D]
        # Permute to [B, world_size, local_seq, local_H, D] then reshape to [B, full_seq, local_H, D]
        output = output.permute(1, 0, 2, 3, 4).contiguous()
        output = output.reshape(B, world_size * local_seq, local_H, D)
        return output

    @staticmethod
    def backward(ctx, grad_output):
        return AllToAllHeadsToSeq.apply(grad_output, ctx.world_size), None


class AllToAllHeadsToSeq(torch.autograd.Function):
    """Reshard from [B, full_seq, local_H, D] to [B, local_seq, H, D] via all_to_all."""
    @staticmethod
    def forward(ctx, x, world_size):
        ctx.world_size = world_size
        B, full_seq, local_H, D = x.shape
        local_seq = full_seq // world_size

        # Reshape to [B, world_size, local_seq, local_H, D]
        x = x.reshape(B, world_size, local_seq, local_H, D)
        # Permute to [world_size, B, local_seq, local_H, D]
        x = x.permute(1, 0, 2, 3, 4).contiguous()

        output = torch.empty_like(x)
        input_list = list(x.unbind(0))
        output_list = list(output.unbind(0))
        dist.all_to_all(output_list, input_list)
        output = torch.stack(output_list, dim=0)

        # output: [world_size, B, local_seq, local_H, D]
        # Permute to [B, local_seq, world_size, local_H, D] then reshape to [B, local_seq, H, D]
        output = output.permute(1, 2, 0, 3, 4).contiguous()
        output = output.reshape(B, local_seq, world_size * local_H, D)
        return output

    @staticmethod
    def backward(ctx, grad_output):
        return AllToAllSeqToHeads.apply(grad_output, ctx.world_size), None


class SPAttention(nn.Module):
    """Sequence-parallel attention using sdpa_attention_forward."""
    def __init__(self, config, world_size):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
        self.num_key_value_groups = self.num_heads // self.num_kv_heads
        self.scaling = self.head_dim ** -0.5
        self.world_size = world_size
        self.local_num_heads = self.num_heads // world_size
        self.local_num_kv_heads = self.num_kv_heads // world_size

        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)

    def forward(self, hidden_states, position_embeddings, attention_mask=None):
        """hidden_states: [B, local_seq, hidden_size]"""
        B, local_seq, _ = hidden_states.shape

        # QKV on local tokens -> [B, local_seq, num_heads/num_kv_heads, head_dim]
        q = self.q_proj(hidden_states).view(B, local_seq, self.num_heads, self.head_dim)
        k = self.k_proj(hidden_states).view(B, local_seq, self.num_kv_heads, self.head_dim)
        v = self.v_proj(hidden_states).view(B, local_seq, self.num_kv_heads, self.head_dim)

        # all_to_all: [B, local_seq, H, D] -> [B, full_seq, local_H, D]
        q = AllToAllSeqToHeads.apply(q, self.world_size)
        k = AllToAllSeqToHeads.apply(k, self.world_size)
        v = AllToAllSeqToHeads.apply(v, self.world_size)

        # -> [B, local_heads, full_seq, D]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # RoPE on full sequence positions
        cos, sin = position_embeddings
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        # sdpa_attention_forward needs self.num_key_value_groups for GQA repeat_kv
        attn_output, attn_weights = sdpa_attention_forward(
            self, q, k, v, attention_mask,
            dropout=0.0,
            scaling=self.scaling,
            is_causal=attention_mask is None,
        )
        # attn_output: [B, full_seq, local_heads * head_dim]
        # Reshape to [B, full_seq, local_heads, head_dim] for all_to_all back
        attn_output = attn_output.view(B, -1, self.local_num_heads, self.head_dim)

        # all_to_all back: [B, full_seq, local_H, D] -> [B, local_seq, H, D]
        attn_output = AllToAllHeadsToSeq.apply(attn_output, self.world_size)

        # [B, local_seq, all_heads * head_dim]
        attn_output = attn_output.reshape(B, local_seq, -1)
        return self.o_proj(attn_output)


class SPDecoderLayer(nn.Module):
    def __init__(self, config, world_size):
        super().__init__()
        self.input_layernorm = LlamaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = LlamaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.self_attn = SPAttention(config, world_size)
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)

    def forward(self, hidden_states, position_embeddings, attention_mask=None):
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = self.self_attn(hidden_states, position_embeddings, attention_mask)
        hidden_states = residual + hidden_states

        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.down_proj(F.silu(self.gate_proj(hidden_states)) * self.up_proj(hidden_states))
        hidden_states = residual + hidden_states
        return hidden_states


class SPLlamaModel(nn.Module):
    def __init__(self, config, rank, world_size):
        super().__init__()
        self.config = config
        self.rank = rank
        self.world_size = world_size
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.norm = LlamaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.rotary_emb = LlamaRotaryEmbedding(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.layers = nn.ModuleList([
            SPDecoderLayer(config, world_size) for _ in range(config.num_hidden_layers)
        ])

    def forward(self, input_ids=None, inputs_embeds=None):
        if inputs_embeds is None:
            full_embeds = self.embed_tokens(input_ids)
        else:
            full_embeds = inputs_embeds

        B, full_seq, D = full_embeds.shape
        local_seq = full_seq // self.world_size
        assert full_seq % self.world_size == 0

        # Each rank takes its token shard
        start = self.rank * local_seq
        hidden_states = full_embeds[:, start:start + local_seq, :]

        # RoPE for full sequence
        position_ids = torch.arange(full_seq, device=hidden_states.device).unsqueeze(0)
        position_embeddings = self.rotary_emb(hidden_states, position_ids=position_ids)

        for layer in self.layers:
            hidden_states = layer(hidden_states, position_embeddings)

        hidden_states = self.norm(hidden_states)

        # Gather all token hidden states for logits
        all_hidden = [torch.zeros_like(hidden_states) for _ in range(self.world_size)]
        dist.all_gather(all_hidden, hidden_states)
        full_hidden = torch.cat(all_hidden, dim=1)
        return self.lm_head(full_hidden)


def load_sp_model(model_name, rank, world_size):
    config = LlamaConfig.from_pretrained(model_name)
    ref_model = transformers.LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
    sp_model = SPLlamaModel(config, rank, world_size)

    with torch.no_grad():
        sp_model.embed_tokens.weight.copy_(ref_model.model.embed_tokens.weight)
        sp_model.norm.weight.copy_(ref_model.model.norm.weight)
        sp_model.lm_head.weight.copy_(ref_model.lm_head.weight)

        for layer_idx in range(config.num_hidden_layers):
            ref_layer = ref_model.model.layers[layer_idx]
            sp_layer = sp_model.layers[layer_idx]

            sp_layer.input_layernorm.weight.copy_(ref_layer.input_layernorm.weight)
            sp_layer.post_attention_layernorm.weight.copy_(ref_layer.post_attention_layernorm.weight)
            sp_layer.self_attn.q_proj.weight.copy_(ref_layer.self_attn.q_proj.weight)
            sp_layer.self_attn.k_proj.weight.copy_(ref_layer.self_attn.k_proj.weight)
            sp_layer.self_attn.v_proj.weight.copy_(ref_layer.self_attn.v_proj.weight)
            sp_layer.self_attn.o_proj.weight.copy_(ref_layer.self_attn.o_proj.weight)
            sp_layer.gate_proj.weight.copy_(ref_layer.mlp.gate_proj.weight)
            sp_layer.up_proj.weight.copy_(ref_layer.mlp.up_proj.weight)
            sp_layer.down_proj.weight.copy_(ref_layer.mlp.down_proj.weight)

    del ref_model
    return sp_model, config


if __name__ == "__main__":
    dist.init_process_group("gloo")
    torch.manual_seed(1337)
    rank, world_size = dist.get_rank(), dist.get_world_size()

    MODEL_NAME = "unsloth/Llama-3.2-1B"
    tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

    if rank == 0:
        ref_model = transformers.LlamaForCausalLM.from_pretrained(
            MODEL_NAME, torch_dtype=torch.float32, attn_implementation="sdpa",
        )
        ref_model.eval()
    dist.barrier()

    for active_rank in range(world_size):
        dist.barrier()
        if rank != active_rank:
            continue
        sp_model, config = load_sp_model(MODEL_NAME, rank, world_size)
        sp_model.eval()
        print(f"Initialized SP model on {rank=}", flush=True)
    dist.barrier()

    # Use sequence divisible by world_size
    prompt = "The quick brown fox jumps over the lazy dog and then runs fast across the wide open field"
    input_ids = tokenizer(prompt, return_tensors='pt')["input_ids"]
    target_len = (input_ids.shape[1] // world_size) * world_size
    if target_len == 0:
        target_len = world_size
    input_ids = input_ids[:, :target_len]

    # Test 1: Forward
    with torch.no_grad():
        sp_logits = sp_model(input_ids=input_ids)

    if rank == 0:
        with torch.no_grad():
            ref_logits = ref_model(input_ids).logits
        max_diff = (sp_logits - ref_logits).abs().max()
        print(f"\nForward pass:")
        print(f"  Ref logits (last, first 5): {ref_logits[0, -1, :5]}")
        print(f"  SP  logits (last, first 5): {sp_logits[0, -1, :5]}")
        print(f"  Max diff: {max_diff}")
        assert torch.allclose(sp_logits, ref_logits, atol=1e-2), f"Logits mismatch! Max diff: {max_diff}"
        print("  PASSED", flush=True)

    # Test 2: Backward
    dist.barrier()
    if rank == 0:
        ref_embeds = ref_model.model.embed_tokens(input_ids).detach().requires_grad_(True)
        ref_out = ref_model.model(inputs_embeds=ref_embeds, use_cache=False).last_hidden_state
        ref_model.lm_head(ref_out).sum().backward()
        ref_embed_grad = ref_embeds.grad.clone()

    sp_embeds = sp_model.embed_tokens(input_ids).detach().requires_grad_(True)
    sp_model(inputs_embeds=sp_embeds).sum().backward()

    if rank == 0:
        max_grad_diff = (sp_embeds.grad - ref_embed_grad).abs().max()
        print(f"\nBackward pass:")
        print(f"  Ref embed grad norm: {ref_embed_grad.norm():.4f}")
        print(f"  SP  embed grad norm: {sp_embeds.grad.norm():.4f}")
        print(f"  Max grad diff: {max_grad_diff}")
        assert torch.allclose(sp_embeds.grad, ref_embed_grad, atol=5e-1), f"Grad mismatch! Max diff: {max_grad_diff}"
        print("  PASSED", flush=True)
        print("\nAll sequence parallelism forward tests passed!", flush=True)
        del ref_model


Writing sequence_parallel_forward.py


In [ ]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 sequence_parallel_forward.py

__Task 5 (1 point):__ Now use the script above to parallelize the entire training run. You are free to use other fine-tuning methods (e.g. LoRA or even full fine-tuning), as long as you can demonstrate that the loss goes down.

**Make sure you increase SEQUENCE_LENGTH as much as possible!** Even on a single GPU, you should be able to go into thousands, if not tens of thousands of tokens - and report the maximum sequence length with one and with multiple GPUs respectively.

If you don't have access to multiple GPUs, you may optionally submit a version that does training on a single GPU, but computes attention heads sequentially with gradient checkpointing - but if you do, please announce that you are using this option in bold, capital letters, so that the grader will notice it.

In [2]:
%%writefile sequence_parallel_train.py
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import transformers
from transformers.models.llama.modeling_llama import (
    LlamaConfig, LlamaRotaryEmbedding, LlamaRMSNorm,
    apply_rotary_pos_emb,
)
from transformers.integrations.sdpa_attention import sdpa_attention_forward
from sequence_parallel_forward import (
    AllToAllSeqToHeads, AllToAllHeadsToSeq, SPAttention, SPDecoderLayer,
)


class SPLlamaForTraining(nn.Module):
    """Sequence-parallel Llama with prompt tuning for training."""
    def __init__(self, config, rank, world_size, num_virtual_tokens=32):
        super().__init__()
        self.config = config
        self.rank = rank
        self.world_size = world_size
        self.num_virtual_tokens = num_virtual_tokens

        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.norm = LlamaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.rotary_emb = LlamaRotaryEmbedding(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.layers = nn.ModuleList([
            SPDecoderLayer(config, world_size) for _ in range(config.num_hidden_layers)
        ])
        # Prompt tuning: learnable virtual token embeddings
        self.prompt_embeddings = nn.Embedding(num_virtual_tokens, config.hidden_size)
        nn.init.normal_(self.prompt_embeddings.weight, std=0.02)

    def forward(self, input_ids, labels=None):
        B = input_ids.shape[0]
        token_embeds = self.embed_tokens(input_ids)

        # Prepend prompt embeddings
        prompt_tokens = self.prompt_embeddings.weight.unsqueeze(0).expand(B, -1, -1)
        full_embeds = torch.cat([prompt_tokens, token_embeds], dim=1)

        full_seq = full_embeds.shape[1]
        local_seq = full_seq // self.world_size
        assert full_seq % self.world_size == 0, f"Total seq ({full_seq}) must be divisible by {self.world_size}"

        start = self.rank * local_seq
        hidden_states = full_embeds[:, start:start + local_seq, :]

        position_ids = torch.arange(full_seq, device=hidden_states.device).unsqueeze(0)
        position_embeddings = self.rotary_emb(hidden_states, position_ids=position_ids)

        for layer in self.layers:
            hidden_states = layer(hidden_states, position_embeddings)

        hidden_states = self.norm(hidden_states)

        # Gather all hidden states for logits / loss
        all_hidden = [torch.zeros_like(hidden_states) for _ in range(self.world_size)]
        dist.all_gather(all_hidden, hidden_states)
        full_hidden = torch.cat(all_hidden, dim=1)
        logits = self.lm_head(full_hidden)

        loss = None
        if labels is not None:
            # Skip prompt tokens, shift by 1
            shift_logits = logits[:, self.num_virtual_tokens:-1, :].contiguous()
            shift_labels = labels.contiguous()
            min_len = min(shift_logits.shape[1], shift_labels.shape[1])
            shift_logits = shift_logits[:, :min_len]
            shift_labels = shift_labels[:, :min_len]
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.shape[-1]), shift_labels.view(-1))

        return loss, logits


def load_sp_training_model(model_name, rank, world_size, num_virtual_tokens=32):
    config = LlamaConfig.from_pretrained(model_name)
    ref_model = transformers.LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)

    sp_model = SPLlamaForTraining(config, rank, world_size, num_virtual_tokens)

    with torch.no_grad():
        sp_model.embed_tokens.weight.copy_(ref_model.model.embed_tokens.weight)
        sp_model.norm.weight.copy_(ref_model.model.norm.weight)
        sp_model.lm_head.weight.copy_(ref_model.lm_head.weight)

        for layer_idx in range(config.num_hidden_layers):
            ref_layer = ref_model.model.layers[layer_idx]
            sp_layer = sp_model.layers[layer_idx]

            sp_layer.input_layernorm.weight.copy_(ref_layer.input_layernorm.weight)
            sp_layer.post_attention_layernorm.weight.copy_(ref_layer.post_attention_layernorm.weight)
            sp_layer.self_attn.q_proj.weight.copy_(ref_layer.self_attn.q_proj.weight)
            sp_layer.self_attn.k_proj.weight.copy_(ref_layer.self_attn.k_proj.weight)
            sp_layer.self_attn.v_proj.weight.copy_(ref_layer.self_attn.v_proj.weight)
            sp_layer.self_attn.o_proj.weight.copy_(ref_layer.self_attn.o_proj.weight)
            sp_layer.gate_proj.weight.copy_(ref_layer.mlp.gate_proj.weight)
            sp_layer.up_proj.weight.copy_(ref_layer.mlp.up_proj.weight)
            sp_layer.down_proj.weight.copy_(ref_layer.mlp.down_proj.weight)

    del ref_model
    return sp_model, config


if __name__ == "__main__":
    dist.init_process_group("gloo")
    torch.manual_seed(1337)
    rank, world_size = dist.get_rank(), dist.get_world_size()

    MODEL_NAME = "unsloth/Llama-3.2-1B"
    NUM_VIRTUAL_TOKENS = 32
    SEQUENCE_LENGTH = 512  # Increase as much as your hardware allows!

    tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

    for active_rank in range(world_size):
        dist.barrier()
        if rank != active_rank:
            continue
        sp_model, config = load_sp_training_model(MODEL_NAME, rank, world_size, NUM_VIRTUAL_TOKENS)
        print(f"Loaded SP training model on {rank=}", flush=True)
    dist.barrier()

    # Freeze all except prompt embeddings
    for name, param in sp_model.named_parameters():
        if "prompt_embeddings" not in name:
            param.requires_grad = False

    # Ensure total (prompt + input) is divisible by world_size
    while (SEQUENCE_LENGTH + NUM_VIRTUAL_TOKENS) % world_size != 0:
        SEQUENCE_LENGTH -= 1
    total_seq = SEQUENCE_LENGTH + NUM_VIRTUAL_TOKENS

    # Download text
    if rank == 0:
        os.system("wget -q https://www.gutenberg.org/cache/epub/4300/pg4300.txt -O ulysses.txt 2>/dev/null || true")
    dist.barrier()

    text = open("ulysses.txt").read()
    input_ids = tokenizer(text, return_tensors='pt')['input_ids']
    if rank == 0:
        print(f"Text tokens: {input_ids.shape[1]}, using {SEQUENCE_LENGTH}")
        print(f"Total with prompt: {total_seq}, per rank: {total_seq // world_size}")

    input_ids = input_ids[:, :SEQUENCE_LENGTH]
    labels = input_ids[:, 1:].clone()
    input_ids = input_ids[:, :-1]
    # Re-adjust: now input_ids has SEQUENCE_LENGTH-1 tokens, total = SEQUENCE_LENGTH-1 + NUM_VIRTUAL_TOKENS
    # Ensure divisibility again
    actual_input_len = input_ids.shape[1]
    total_seq = actual_input_len + NUM_VIRTUAL_TOKENS
    while total_seq % world_size != 0:
        actual_input_len -= 1
        total_seq = actual_input_len + NUM_VIRTUAL_TOKENS
    input_ids = input_ids[:, :actual_input_len]
    labels = labels[:, :actual_input_len]

    # Wrap with DDP to sync prompt_embeddings gradients across ranks
    sp_model = DDP(sp_model, find_unused_parameters=True)

    trainable_params = [p for p in sp_model.parameters() if p.requires_grad]
    total_trainable = sum(p.numel() for p in trainable_params)
    total_params = sum(p.numel() for p in sp_model.parameters())
    if rank == 0:
        print(f"Parameters: {total_trainable} trainable / {total_params} total")

    opt = torch.optim.Adam(trainable_params, lr=1e-3)

    for i in range(10):
        # Call through DDP wrapper (not module.forward) for gradient sync
        loss, logits = sp_model(input_ids, labels=labels)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if rank == 0:
            print(f"  {i=}\t{loss.item()=:.4f}", flush=True)

    if rank == 0:
        print(f"\nTraining complete! Sequence length: {actual_input_len}")
        print(f"With {world_size} ranks, each processes {total_seq // world_size} tokens/step")
        print("All sequence parallelism training tests passed!")


Writing sequence_parallel_train.py


In [ ]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 sequence_parallel_train.py

```

```

```

```

```

```


### Optional: bonus tasks

There are many routes to further improve the training/inference code. You may (but you don't have to) implement any combination of them for bonus points.

However, please not that the total points for this week's entire assignment (part 1 & 2) are **capped at 14**.

__Bonus task: parallel key-value caching (1 point).__ In tasks 2 and 3, you implement tensor parallelism for attention forward pass and perform inference with re-computation. However, real world inference engines use [KV caching](https://huggingface.co/docs/transformers/main/en/kv_cache) - keeping key and value caches from past tokens and only processing the new token each time.

For this task, you will have to implement this type of parallelism for either torch.distributed or DTensor implementation of attention $-$ simply cache the heads already assigned to each rank. To get the grade, you will need to demonstrate that the model generates a sensible text with any cache (via past_key_values=).

__Bonus task: pipeline parallelism (1-2 points):__ In tasks 1-3, you've implemented symmetric model parallelism, aka Tensor Parallelism. However, there is another way to partition model parameters $-$ assign entire layers to each rank and run them in a pipeline. This can be faster, especially if you are running

For 1 point, check out [torch.distributed.pipelinging](https://pytorch.org/docs/stable/distributed.pipelining.html), [DeepSpeed pipelining](https://deepspeed.readthedocs.io/en/latest/pipeline.html) or [torchgpipe](https://github.com/kakaobrain/torchgpipe) and demonstrate that you can run or fine-tune a model that would not fit into a single GPU (you will need multuple devices for this!).

For 2 points, compare different pipelining schedules in terms of training throughput: use GPipe as a baseline and try ScheduleInterleaved1F1B (or a more advanced pipeline of your choosing).

__Bonus task: better sequence parallelism (2 points).__ In tasks 4 and 5, you implemented basic sequence parallelism. However, there are multiple ways you can improve that technique for further memory savings or better device utilization.

For 1 point, implement combined tensor + sequence parallelism and compare results with naive sequence parallelism.

For 2 points, implement [Ring Attention](https://arxiv.org/abs/2310.01889) *or* integrate computation-communication overlap from [FLUX](https://arxiv.org/abs/2406.06858) and measure the speed and memory trade-offs.